In [1]:
import pandas as pd
import numpy as np

In [2]:
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery


### 1. Weekly Cohort Retention

In [3]:
orders["order_week"] = orders["order_ts"].dt.to_period("W").dt.start_time ## Date Trunc equivalent
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_week
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy,2024-12-30
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy,2025-01-06
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy,2025-01-06
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy,2025-01-06
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery,2025-01-06


In [4]:
orders.dtypes

order_id                 int64
user_id                  int64
store_id                   str
city                       str
order_ts        datetime64[us]
items                    int64
gmv                    float64
delivery_min           float64
category                   str
order_week      datetime64[us]
dtype: object

In [5]:
user_order_weeks = orders[['user_id', 'order_week']].drop_duplicates()
user_order_weeks["user_first_week"] = (
    user_order_weeks
    .groupby(["user_id"])["order_week"]
    .transform("min")
)

user_order_weeks["week_diff"] = (
    user_order_weeks["order_week"] - user_order_weeks["user_first_week"]
)/pd.Timedelta(weeks=1)

user_order_weeks["week_diff"] = user_order_weeks["week_diff"].astype(int)

user_order_weeks.head()

,user_id,order_week,user_first_week,week_diff
0,10700,2024-12-30,2024-12-30,0
1,8110,2025-01-06,2025-01-06,0
2,15650,2025-01-06,2025-01-06,0
3,1908,2025-01-06,2025-01-06,0
4,2751,2025-01-06,2025-01-06,0


In [6]:
pivot_df = pd.pivot_table(
    user_order_weeks[user_order_weeks['week_diff'] <= 12],
    index = "user_first_week",
    columns = "week_diff",
    values = "user_id",
    aggfunc = "count"
)

last_week = pivot_df.index.max()

# Calculate the latest observable week for each cohort
latest_cohort_week = pivot_df.index.max()

for cohort_week in pivot_df.index:
    max_week_diff = (
        (latest_cohort_week - cohort_week).days // 7
    )

    cols_to_fill = [
        col for col in pivot_df.columns
        if isinstance(col, (int, float, np.integer, np.floating))
        and col <= max_week_diff
    ]

    pivot_df.loc[cohort_week, cols_to_fill] = (
        pivot_df.loc[cohort_week, cols_to_fill].fillna(0)
    )

pivot_df.insert(0, "cohort_size", pivot_df[0])

pivot_df.iloc[:, 1:] = pivot_df.iloc[:, 1:].div(pivot_df[0], axis=0) * 100

pivot_df

week_diff,cohort_size,0,1,2,3,4,5,6,7,8,9,10,11,12
user_first_week,,,,,,,,,,,,,,
2024-12-30,1.0,100.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000
2025-01-06,16.0,100.0,0.000000,6.250000,0.000000,0.000000,12.500000,18.750000,0.000000,6.250000,6.250000,0.000000,6.250000,0.000000
2025-01-13,27.0,100.0,7.407407,14.814815,14.814815,7.407407,11.111111,0.000000,18.518519,3.703704,3.703704,11.111111,7.407407,7.407407
2025-01-20,37.0,100.0,10.810811,8.108108,13.513514,10.810811,10.810811,2.702703,5.405405,8.108108,16.216216,16.216216,5.405405,2.702703
2025-01-27,48.0,100.0,2.083333,4.166667,6.250000,8.333333,2.083333,4.166667,8.333333,8.333333,4.166667,8.333333,4.166667,4.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-31,89.0,100.0,8.988764,6.741573,11.235955,1.123596,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-07,85.0,100.0,4.705882,4.705882,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-14,75.0,100.0,2.666667,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. 7-day rolling GMV per city

In [7]:
orders["order_date"] = orders["order_ts"].dt.normalize()

city_summary = (
    orders
    .groupby(["city"], as_index = False)
    .agg(
        first_order_date = ("order_date", "min"),
        last_order_date = ("order_date", "max")
    )
)

city_order_dates = pd.concat(
    city_summary.apply(
        lambda row: pd.DataFrame({
            "city": row["city"],
            "order_date": pd.date_range(
                row["first_order_date"],
                row["last_order_date"],
                freq="D"
            )
        }),
        axis=1
    ).tolist(),
    ignore_index=True
)

daily_city_summary = (
    orders
    .groupby(["order_date", "city"], as_index = False)
    .agg(
        gmv = ("gmv", "sum")
    )
)

daily_city_summary = city_order_dates.merge(
    daily_city_summary,
    on = ["city", "order_date"],
    how = "left"
)

daily_city_summary["gmv"] = daily_city_summary["gmv"].fillna(0)

daily_city_summary["rolling_7d_gmv"] = (
    daily_city_summary
    .groupby("city")["gmv"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

daily_city_summary

,city,order_date,gmv,rolling_7d_gmv
0,Bengaluru,2025-01-08,199.17,NaN
1,Bengaluru,2025-01-09,0.00,NaN
2,Bengaluru,2025-01-10,0.00,NaN
3,Bengaluru,2025-01-11,164.65,NaN
4,Bengaluru,2025-01-12,573.28,NaN
...,...,...,...,...
3783,Pune,2026-09-25,14823.65,109576.38
3784,Pune,2026-09-26,12700.97,111703.24
3785,Pune,2026-09-27,9970.62,106610.41
3786,Pune,2026-09-28,6288.67,95112.12


Here we have created a base of order dates and cities. If we hadn't created that, we would need to use rolling("7D"). Final outcome would only change where there is no order date present in a city. Since, order date is missing, final output will miss that date. But where dates are present, rolling("7D") would have worked same for SUM

### 3. Order Sequence per User

In [8]:
orders = orders.sort_values(
    by = ["user_id", "order_ts"],
    ascending = [True, True]
)

orders["order_number"] = (
    orders
    .groupby(["user_id"])["order_ts"]
    .rank(
        method = "first",
        ascending = True
    )
)

orders["prev_order_ts"] = (
    orders
    .groupby("user_id")["order_ts"]
    .shift(1)
)

orders["days_since_prev_order"] = (
    orders["order_ts"] - orders["prev_order_ts"]
).dt.days

orders["prev_order_gmv"] = (
    orders
    .groupby("user_id")["gmv"]
    .shift(1)
)

orders["is_higher_gmv_than_prev_order"] = np.where(
    orders["prev_order_gmv"].isnull(),
    pd.NA,
    orders["gmv"] > orders["prev_order_gmv"]
)

orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_week,order_date,order_number,prev_order_ts,days_since_prev_order,prev_order_gmv,is_higher_gmv_than_prev_order
56929,56930,1,BEN-08,Bengaluru,2026-06-04 14:33:00,4,188.05,7.1,grocery,2026-06-01,2026-06-04,1.0,NaT,NaN,NaN,<NA>
63978,63979,1,BEN-01,Bengaluru,2026-06-26 13:14:00,2,78.40,18.7,household,2026-06-22,2026-06-26,2.0,2026-06-04 14:33:00,21.0,188.05,False
70176,70177,1,BEN-04,Bengaluru,2026-07-14 18:08:00,5,539.01,14.8,pharmacy,2026-07-13,2026-07-14,3.0,2026-06-26 13:14:00,18.0,78.40,True
75146,75147,1,BEN-02,Bengaluru,2026-07-29 14:42:00,8,366.25,13.1,personal_care,2026-07-27,2026-07-29,4.0,2026-07-14 18:08:00,14.0,539.01,False
31721,31722,2,PUN-06,Pune,2026-02-18 13:06:00,5,313.31,17.0,grocery,2026-02-16,2026-02-18,1.0,NaT,NaN,NaN,<NA>


In [9]:
second_orders = orders[orders["order_number"] == 2]

second_orders["days_since_prev_order"].median()

np.float64(38.0)